# 실습 20. ICER와 순편익 계산

사회약학 연구방법 노트의 '실습 20 ICER와 순편익 계산'에 나오는 셀을 차례로 모은 노트북입니다. 진행성 신세포암 1차 치료(신약 A 대 표준요법 B, 가상의 예시)의 기준 분석 결과와 20장이 지어낸 대안 두 개(기존약 C, 병용요법 D)로 증분비용효과비, 지배와 확장 지배, 효율 경계, 순금전편익과 순건강편익, 비용효과평면을 계산해 20장의 숫자와 맞춰 봅니다. 자료 파일 없이 숫자를 직접 적으므로 인터넷 연결이 없어도 됩니다. 설명과 출력 읽는 법은 사이트의 실습 20 쪽에 있습니다.


## a. 실습 데이터 준비

출력을 만든 패키지의 버전을 확인하고, 네 대안(표준요법 B, 기존약 C, 병용요법 D, 신약 A)의 총비용과 QALY를 표로 만듭니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", matplotlib.__version__)

### 셀 2. 네 대안의 총비용과 QALY

In [ ]:
alts = pd.DataFrame({
    "cost": [7689.485, 8100.0, 9400.0, 10561.778],   # 총비용(만원, 할인 후)
    "qaly": [1.873640, 1.700, 2.000, 2.384669],       # 총 QALY(할인 후)
}, index=["B", "C", "D", "A"])                         # B 표준요법, C 기존약, D 병용요법, A 신약
print(alts.shape)
alts

## b. 증분비용효과비와 지배

신약 A 대 표준요법 B의 ICER를 구하고, 사분면 판정과 효율 경계를 코드로 만듭니다.

### 셀 3. 신약 A 대 표준요법 B의 증분과 ICER

In [ ]:
cost_A, qaly_A = alts.loc["A", "cost"], alts.loc["A", "qaly"]
cost_B, qaly_B = alts.loc["B", "cost"], alts.loc["B", "qaly"]

d_cost = cost_A - cost_B               # 증분비용
d_qaly = qaly_A - qaly_B               # 증분 QALY
icer = d_cost / d_qaly                 # 증분비용효과비
print("증분비용  :", round(d_cost, 1), "만원")
print("증분 QALY :", round(d_qaly, 3))
print("ICER      :", round(icer), "만원/QALY")
print("평균 비   : A", round(cost_A / qaly_A), " B", round(cost_B / qaly_B))

ly_A, ly_B = 3.398772, 2.728843        # 할인한 생존연수
print("생존연수당:", round(d_cost / (ly_A - ly_B)), "만원/생존연수")

### 셀 4. 사분면 판정과 음수 ICER

In [ ]:
def quadrant(dc, dq):
    """증분비용(dc)과 증분 QALY(dq)의 부호로 비용효과평면의 사분면을 판정"""
    if dq > 0 and dc <= 0:
        return "오른쪽 아래, 우월(dominant)"
    if dq <= 0 and dc > 0:
        return "왼쪽 위, 열등(dominated)"
    if dq > 0:
        return "오른쪽 위, ICER를 임계값과 비교함"
    return "왼쪽 아래, ICER가 임계값보다 높아야 받아들임"

cases = {"A vs B": (d_cost, d_qaly),
         "예 1  ": (-500, 0.2),       # 500만원 덜 들고 0.2 QALY를 더 얻음
         "예 2  ": (500, -0.2),       # 500만원 더 들고 0.2 QALY를 잃음
         "예 3  ": (-1000, -0.1)}     # 1,000만원 아끼고 0.1 QALY를 잃음
for name, (dc, dq) in cases.items():
    print(name, "ICER", round(dc / dq), "|", quadrant(dc, dq))

### 셀 5. 열등, 확장 열등, 효율 경계

In [ ]:
def icer_between(df, lo, hi):
    """대안 lo에서 대안 hi로 옮길 때의 ICER"""
    return ((df.loc[hi, "cost"] - df.loc[lo, "cost"])
            / (df.loc[hi, "qaly"] - df.loc[lo, "qaly"]))

def frontier(df):
    """비용순으로 놓고 열등과 확장 열등을 표시한 뒤, 효율 경계 위에서 이웃끼리 ICER를 구한 표"""
    df = df.sort_values(["cost", "qaly"], ascending=[True, False]).copy()  # 비용이 낮은 순서, 같으면 효과가 큰 것부터
    df["status"], df["vs"], df["icer"] = "경계", "", np.nan
    keep = []                                   # 경계에 남은 대안의 이름
    for i in df.index:
        if any(df.loc[j, "qaly"] >= df.loc[i, "qaly"] for j in keep):
            df.loc[i, "status"] = "열등"         # 비용이 같거나 더 싸면서 효과가 같거나 큰 대안이 있음
            continue
        keep.append(i)
        while len(keep) >= 3:                   # 이어진 세 대안의 ICER 순서 확인
            a, b, c = keep[-3:]
            if icer_between(df, a, b) <= icer_between(df, b, c):
                break
            print(b, "확장 열등:", round(icer_between(df, a, b)), ">",
                  round(icer_between(df, b, c)))
            df.loc[b, "status"] = "확장 열등"
            keep.pop(-2)                        # 가운데 대안을 뺌
    for a, b in zip(keep[:-1], keep[1:]):       # 경계 위의 이웃끼리
        df.loc[b, "vs"] = a
        df.loc[b, "icer"] = icer_between(df, a, b)
    return df

front = frontier(alts)
front.round({"icer": 0})

### 셀 6. B와 A를 섞은 전략과 병용요법 D

In [ ]:
f = (alts.loc["D", "qaly"] - qaly_B) / (qaly_A - qaly_B)   # A를 쓸 환자의 비율
mix_cost = cost_B + f * (cost_A - cost_B)
mix_qaly = qaly_B + f * (qaly_A - qaly_B)
print("A를 쓰는 비율:", round(f * 100, 1), "%")
print("섞은 전략 : QALY", round(mix_qaly, 3), " 비용", round(mix_cost, 1))
print("병용요법 D: QALY", alts.loc["D", "qaly"], " 비용", alts.loc["D", "cost"])
print("D보다 적게 드는 비용:", round(alts.loc["D", "cost"] - mix_cost, 1), "만원")

### 셀 7. 네 대안과 효율 경계

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
on = front[front["status"] == "경계"]
off = front[front["status"] != "경계"]
ax.plot(on["qaly"], on["cost"], "-o", color="tab:blue", label="Efficiency frontier")
ax.plot(off["qaly"], off["cost"], "s", color="tab:red",
        label="Dominated or extendedly dominated")
ax.plot(mix_qaly, mix_cost, "o", mfc="white", color="tab:blue",
        label="24.7% A + 75.3% B")
for name, row in front.iterrows():
    ax.annotate(name, (row["qaly"], row["cost"]),
                xytext=(6, -12), textcoords="offset points")
ax.margins(x=0.08, y=0.12)              # 가장자리의 이름이 잘리지 않게
ax.set_xlabel("Total QALYs")
ax.set_ylabel("Total cost (10,000 KRW)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()

## c. 순편익과 비용효과평면

이 예시에서 가정한 임계값 5,000만원/QALY(우리나라에는 공식 임계값이 없음)에서 순금전편익과 순건강편익을 구하고, 임계값을 바꿔 봅니다.

### 셀 8. 임계값 5,000만원에서 순금전편익과 순건강편익

In [ ]:
lam = 5000                             # 이 예시에서 가정한 임계값(만원/QALY)
nmb = lam * d_qaly - d_cost            # 증분 순금전편익(만원)
nhb = d_qaly - d_cost / lam            # 증분 순건강편익(QALY)
print("순금전편익:", round(nmb, 1), "만원")
print("순건강편익:", round(nhb, 3), "QALY")
print("환자 100명이면:", round(100 * nhb, 1), "QALY")
print("ICER < 임계값:", icer < lam, "  순금전편익 > 0:", nmb > 0)

### 셀 9. 임계값을 바꾼 순금전편익

In [ ]:
tab = pd.DataFrame({"lam": [2500, 3000, 5000, 6000, 7000, 8000]})
tab["nmb"] = tab["lam"] * d_qaly - d_cost      # 순금전편익(만원)
tab["nhb"] = tab["nmb"] / tab["lam"]           # 순건강편익(QALY)
tab["ce"] = tab["nmb"] > 0                     # 비용효과적인가
print("부호가 바뀌는 임계값:", round(d_cost / d_qaly, 1))
tab.round({"nmb": 1, "nhb": 3})

### 셀 10. 네 대안의 순금전편익

In [ ]:
nmb4 = pd.DataFrame({lam_: lam_ * alts["qaly"] - alts["cost"]
                     for lam_ in [5000, 6000]})
print("순금전편익이 가장 큰 대안:", nmb4.idxmax().to_dict())
nmb4.round(1)

### 셀 11. 임계값마다 순금전편익이 가장 큰 대안

In [ ]:
grid = np.arange(0, 20001, 100)        # 임계값 0, 100, ..., 20,000
best = [(g * alts["qaly"] - alts["cost"]).idxmax() for g in grid]
print("가장 큰 대안이 된 횟수:", pd.Series(best).value_counts().to_dict())
print("A가 처음 가장 커지는 임계값:", grid[best.index("A")])

### 셀 12. 비용효과평면

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
x = np.array([-0.6, 0.8])
ax.fill_between(x, lam * x, -3500, color="tab:blue", alpha=0.08)   # 비용효과적인 영역
ax.plot(x, lam * x, color="gray", ls="--", label="Threshold 5,000 per QALY")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.plot(d_qaly, d_cost, "o", ms=8, color="tab:blue", label="Drug A vs B")
ax.plot([d_qaly, d_qaly], [lam * d_qaly, d_cost], color="tab:red", lw=2,
        label="NMB = -317")
ax.plot(0.2, -500, "^", ms=8, color="tab:green", label="Example 1 (dominant)")
ax.plot(-0.2, 500, "v", ms=8, color="tab:orange", label="Example 2 (dominated)")
ax.set_xlim(-0.6, 0.8)
ax.set_ylim(-3500, 4500)
ax.set_xlabel("Incremental QALYs")
ax.set_ylabel("Incremental cost (10,000 KRW)")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()

## 과제 정답

**과제 1.** 셀 9 표의 임계값 6,000만원 행(순금전편익 +193.9만원)을 급여 평가 보고서에 쓰듯 한 문장으로 옮기고, 이 값이 뜻하지 않는 것을 두 가지 씁니다.

### 셀 13. 과제 1 정답. 임계값 6,000만원 행 풀어 보기

In [ ]:
lam6 = 6000
value = lam6 * d_qaly                  # 늘어난 QALY를 임계값으로 돈으로 바꾼 값
print("늘어난 QALY의 값:", round(value, 1), "만원")
print("늘어난 비용     :", round(d_cost, 1), "만원")
print("순금전편익      :", round(value - d_cost, 1), "만원")
print("순건강편익      :", round(d_qaly - d_cost / lam6, 3), "QALY")
print("환자 100명이면  :", round(100 * (value - d_cost)), "만원,",
      round(100 * (d_qaly - d_cost / lam6), 1), "QALY")

**과제 2.** 표준요법 B보다 600만원 더 들고 0.20 QALY를 더 얻는 신약 E를 네 대안에 더해 효율 경계를 다시 만들고, 임계값 5,000만원과 8,000만원에서 어느 대안을 고르는지 답합니다.

### 셀 14. 과제 2 정답. 신약 E를 더한 효율 경계

In [ ]:
alts_E = alts.copy()
alts_E.loc["E"] = [cost_B + 600, qaly_B + 0.20]   # 신약 E: B보다 600만원 더, 0.20 QALY 더
front_E = frontier(alts_E)
print(front_E.round({"icer": 0}))
for lam_ in [5000, 8000]:
    nb_ = lam_ * alts_E["qaly"] - alts_E["cost"]
    print(lam_, "| 가장 큰 대안:", nb_.idxmax(),
          "| B 대비 E", round(nb_["E"] - nb_["B"], 1),
          " A", round(nb_["A"] - nb_["B"], 1))

**과제 3.** 총비용 ÷ QALY와 병용요법 D와 비교한 ICER로 신약 A가 비용효과적이라고 결론 낸 동료의 분석에서 틀린 곳 두 가지를 찾습니다.

### 셀 15. 과제 3 정답. 동료의 분석 다시 보기

In [ ]:
ratio = alts["cost"] / alts["qaly"]               # 동료가 구한 총비용 ÷ QALY
print("동료의 비:", ratio.round().to_dict())
print("임계값보다 낮은 대안:", list(ratio[ratio < lam].index))
print("동료의 A 대 D:", round(icer_between(alts, "D", "A")))
print("D의 판정:", front.loc["D", "status"])
print("경계 위 A 대", front.loc["A", "vs"], ":", round(front.loc["A", "icer"]))
print("임계값 5,000에서 순금전편익이 가장 큰 대안:",
      (lam * alts["qaly"] - alts["cost"]).idxmax())